In [ ]:
!export KAGGLE_API_TOKEN=KGAT_8e742fd0b13975e80990e57844537985

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
import re, random
from pathlib import Path
from typing import List
from transformers import T5TokenizerFast

TEXT_COL = "review"

OUT_DIR = Path("tokenizer")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CHUNKS = OUT_DIR / "ae_chunks.txt"
OUT_SAMPLES = OUT_DIR / "ae_chunks_samples.txt"

# Chunking config
SENTS_PER_CHUNK = 3
MIN_SENT_LEN_CHARS = 3
MAX_T5_TOKENS = 144
SEED = 42
random.seed(SEED)

BOS = "<bos>"
EOS = "<eos>"

MODEL_NAME = "t5-small"
t5_tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
def clean_html(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

_ABBR = [
    "Mr", "Mrs", "Ms", "Dr", "Prof", "Sr", "Jr",
    "St", "Mt", "Gen", "Col", "Sgt", "Capt",
    "e.g", "i.e", "etc", "vs",
    "U.S", "U.K", "No", "Dept", "Inc", "Ltd",
]
_ABBR_PATTERN = r"(?:%s)" % "|".join(re.escape(a) for a in _ABBR)

def split_sentences(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []
    text = text.replace("...", " <ELLIPSIS> ")

    def _protect_abbr(m):
        return m.group(0).replace(".", "<DOT>")

    text = re.sub(rf"\b{_ABBR_PATTERN}\.", _protect_abbr, text)
    text = re.sub(r"\b([A-Z])\.", r"\1<DOT>", text)
    text = re.sub(r"\b([A-Z])<DOT>\s*([A-Z])<DOT>", r"\1<DOT>\2<DOT>", text)

    parts = re.split(r"(?<=[.!?])\s+(?=(?:[\"'(\[])?[A-Z0-9])", text)

    sents = []
    for p in parts:
        p = p.replace("<DOT>", ".").replace("<ELLIPSIS>", "...")
        p = re.sub(r"\s+", " ", p).strip()
        if len(p) >= MIN_SENT_LEN_CHARS:
            sents.append(p)
    return sents

def t5_token_len(text: str) -> int:
    # No special tokens; T5TokenizerFast will add internally if asked
    return len(t5_tokenizer.encode(text, add_special_tokens=False))

def split_by_t5_tokens(text: str, max_tokens: int) -> List[str]:
    """
    Split a text into subchunks so each subchunk has <= max_tokens
    by T5 token length.
    We do this by splitting at whitespace boundaries to avoid decoding artifacts.
    """
    if t5_token_len(text) <= max_tokens:
        return [text]

    words = text.split()
    out = []
    cur = []
    for w in words:
        trial = " ".join(cur + [w]) if cur else w
        if t5_token_len(trial) <= max_tokens:
            cur.append(w)
        else:
            if cur:
                out.append(" ".join(cur))
            cur = [w]
    if cur:
        out.append(" ".join(cur))

    # Rare case: a single word exceeds max_tokens (shouldn't happen)
    # fallback: hard truncate by token ids
    final = []
    for chunk in out:
        ids = t5_tokenizer.encode(chunk, add_special_tokens=False)
        if len(ids) <= max_tokens:
            final.append(chunk)
        else:
            ids = ids[:max_tokens]
            final.append(t5_tokenizer.decode(ids, skip_special_tokens=True).strip())
    return [c for c in final if c.strip()]

def make_chunks(sentences: List[str], sents_per_chunk: int) -> List[str]:
    chunks = []
    i = 0
    n = len(sentences)
    while i < n:
        base_chunk = " ".join(sentences[i:i + sents_per_chunk]).strip()
        if base_chunk:
            chunks.extend(split_by_t5_tokens(base_chunk, MAX_T5_TOKENS))
        i += sents_per_chunk
    return chunks

# Build chunks
texts = df[TEXT_COL].astype(str).tolist()
texts = [clean_html(t) for t in texts]

all_chunks: List[str] = []
sent_counts, chunk_counts = [], []

for t in texts:
    sents = split_sentences(t)
    sent_counts.append(len(sents))
    chunks = make_chunks(sents, SENTS_PER_CHUNK)
    chunk_counts.append(len(chunks))
    for c in chunks:
        all_chunks.append(f"{BOS} {c} {EOS}")

with open(OUT_CHUNKS, "w", encoding="utf-8") as f:
    for c in all_chunks:
        f.write(c.replace("\n", " ") + "\n")

print("=== AE Dataset Build Summary (T5 token cap) ===")
print(f"Reviews: {len(texts)}")
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg sentences/review: {sum(sent_counts)/max(1,len(sent_counts)):.2f}")
print(f"Avg chunks/review: {sum(chunk_counts)/max(1,len(chunk_counts)):.2f}")
print(f"Wrote: {OUT_CHUNKS.resolve()}")

# Random samples
k = 12
sample_idxs = random.sample(range(len(all_chunks)), k=min(k, len(all_chunks)))
samples = [all_chunks[i] for i in sample_idxs]
with open(OUT_SAMPLES, "w", encoding="utf-8") as f:
    for j, s in enumerate(samples, 1):
        f.write(f"[SAMPLE {j}]\n{s}\n\n")
print(f"Wrote samples: {OUT_SAMPLES.resolve()}")

Token indices sequence length is longer than the specified maximum sequence length for this model (526 > 512). Running this sequence through the model will result in indexing errors


=== AE Dataset Build Summary (T5 token cap) ===
Reviews: 50000
Total chunks: 268958
Avg sentences/review: 11.55
Avg chunks/review: 5.38
Wrote: /content/tokenizer/ae_chunks.txt
Wrote samples: /content/tokenizer/ae_chunks_samples.txt


In [ ]:
# !pip -q install transformers accelerate evaluate sentencepiece

import os, re, random, math, time
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from transformers import (
    T5TokenizerFast,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)


In [ ]:
CHUNKS_PATH = "/content/tokenizer/ae_chunks.txt"
OUT_DIR = "/content/ae_story1_t5tok_bottleneck256"
os.makedirs(OUT_DIR, exist_ok=True)

# Model / training config
MODEL_NAME = "t5-small"
LATENT_DIM = 512
BOTTLENECK_SEQ_LEN = 32
MAX_LEN = 96
BATCH_SIZE = 16
LR = 1e-4
WEIGHT_DECAY = 0.01
EPOCHS = 2
WARMUP_RATIO = 0.03
GRAD_CLIP = 1.0
SEED = 42


TARGET_NOISE_PROB = 0.15
NOISE_WARMUP_STEPS = 4000

# Regularization
Z_NOISE_STD = 0.0
Z_DROPOUT_P = 0.0


MAX_SAMPLES = None

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [ ]:
def strip_bos_eos(line: str) -> str:
    line = line.strip()
    line = re.sub(r"^\s*<bos>\s*", "", line)
    line = re.sub(r"\s*<eos>\s*$", "", line)
    return line.strip()

class ChunkTextDataset(Dataset):
    def __init__(self, path: str, max_samples=None):
        with open(path, "r", encoding="utf-8") as f:
            lines = [strip_bos_eos(l) for l in f if l.strip()]
        # Drop empties after stripping
        lines = [l for l in lines if len(l) > 0]

        if max_samples is not None and max_samples < len(lines):
            random.shuffle(lines)
            lines = lines[:max_samples]

        self.lines = lines

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        return self.lines[idx]

ds = ChunkTextDataset(CHUNKS_PATH, max_samples=MAX_SAMPLES)
len(ds), ds[0][:200]


(268958,
 "One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me. The first thing that struck me about Oz was i")

In [ ]:
ds

In [ ]:
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)


@dataclass
class Collator:
    tokenizer: T5TokenizerFast
    max_len: int = MAX_LEN
    noise_prob: float = 0.0

    def set_noise_prob(self, p: float):
        self.noise_prob = float(max(0.0, min(1.0, p)))

    def __call__(self, batch_texts: List[str]) -> Dict[str, torch.Tensor]:
        enc = self.tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        labels = enc["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        input_ids = enc["input_ids"].clone()
        attn = enc["attention_mask"]

        if self.noise_prob > 0:
            corrupt = (torch.rand_like(input_ids.float()) < self.noise_prob) & (attn == 1)
            corrupt &= (input_ids != self.tokenizer.pad_token_id)

            input_ids[corrupt] = self.tokenizer.pad_token_id

        return {
            "input_ids": input_ids,
            "attention_mask": attn,
            "labels": labels
        }


collate_fn = Collator(tokenizer, max_len=MAX_LEN, noise_prob=0.0)

In [ ]:
from transformers.modeling_outputs import BaseModelOutput
import torch.nn.functional as F

In [ ]:
class BottleneckT5AE(nn.Module):
    """
    Multi-slot bottleneck T5 autoencoder.
    Encoder states -> K pooled slots -> latent slots -> decoder memory slots.
    """
    def __init__(self, model_name: str, latent_dim: int, bottleneck_seq_len: int, pool_slots: int = 16):
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        d_model = self.t5.config.d_model

        self.latent_dim = latent_dim
        self.bottleneck_seq_len = bottleneck_seq_len
        self.pool_slots = pool_slots
        self.d_model = d_model

        # Learned queries for pooling K slots from encoder outputs
        self.pool_queries = nn.Parameter(torch.randn(pool_slots, d_model) * 0.02)
        self.pool_q_proj = nn.Linear(d_model, d_model, bias=False)
        self.pool_k_proj = nn.Linear(d_model, d_model, bias=False)
        self.pool_v_proj = nn.Linear(d_model, d_model, bias=False)

        # Slot-level bottleneck
        self.down = nn.Linear(d_model, latent_dim)
        self.up = nn.Linear(latent_dim, d_model)

        # Map K d_model slots -> decoder memory length M (bottleneck_seq_len)
        self.to_mem = nn.Linear(pool_slots * d_model, bottleneck_seq_len * d_model)

        # Give each decoder memory slot an identity
        self.mem_slot_emb = nn.Parameter(torch.randn(bottleneck_seq_len, d_model) * 0.02)

    def pool_slots_from_encoder(self, h: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        h: [B,L,d], mask: [B,L]
        returns pooled: [B,K,d]
        """
        B, L, d = h.shape
        K = self.pool_slots

        # Q: [B,K,d]
        Q = self.pool_q_proj(self.pool_queries)[None, :, :].expand(B, K, d)

        # K,V: [B,L,d]
        Kh = self.pool_k_proj(h)
        Vh = self.pool_v_proj(h)

        # attn scores: [B,K,L]
        # scores = torch.einsum("bkd,bld->bkl", Q, Kh) / (d ** 0.5)
        # scores = scores.masked_fill(mask[:, None, :] == 0, -1e9)
        # w = torch.softmax(scores, dim=-1)  # [B,K,L]

        scores = torch.einsum("bkd,bld->bkl", Q, Kh) / (d ** 0.5)

        scores = scores.float()
        neg_inf = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask[:, None, :] == 0, neg_inf)

        w = torch.softmax(scores, dim=-1).to(Vh.dtype)
        pooled = torch.einsum("bkl,bld->bkd", w, Vh)
        return pooled


    def encode_to_mem(self, input_ids, attention_mask):
        # Encoder
        enc_out_full = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = enc_out_full.last_hidden_state  # [B,L,d]

        # Multi-slot pooling: [B,K,d]
        pooled = self.pool_slots_from_encoder(h, attention_mask)

        # Down/up per slot: [B,K,latent] -> [B,K,d]
        z_slots = self.down(pooled)  # [B,K,latent]

        # --- latent dropout (forces usage, prevents shortcutting) ---
        if self.training and Z_DROPOUT_P > 0:
            drop_mask = (torch.rand_like(z_slots) > Z_DROPOUT_P).to(z_slots.dtype)
            z_slots = z_slots * drop_mask

        # --- latent noise (smoothness, reduces brittle loops) ---
        if self.training and Z_NOISE_STD > 0:
            z_slots = z_slots + Z_NOISE_STD * torch.randn_like(z_slots)

        # [B,K,latent]
        pooled_rec = self.up(z_slots)       # [B,K,d]

        # Mix K slots into M memory slots
        flat = pooled_rec.reshape(pooled_rec.size(0), -1)        # [B,K*d]
        mem_flat = self.to_mem(flat)                             # [B,M*d]
        mem = mem_flat.view(mem_flat.size(0), self.bottleneck_seq_len, self.d_model)  # [B,M,d]

        # Add memory slot embeddings
        mem = mem + self.mem_slot_emb[None, :, :]

        mem_mask = torch.ones(mem.size(0), mem.size(1), dtype=torch.long, device=mem.device)
        return mem, mem_mask, z_slots

    def forward(self, input_ids, attention_mask, labels=None):
        mem, mem_mask, _ = self.encode_to_mem(input_ids, attention_mask)
        enc_out = BaseModelOutput(last_hidden_state=mem)

        out = self.t5(
            encoder_outputs=enc_out,
            attention_mask=mem_mask,
            labels=labels
        )
        return out




model = BottleneckT5AE(MODEL_NAME, LATENT_DIM, BOTTLENECK_SEQ_LEN).to(device)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def freeze_decoder(model: BottleneckT5AE, freeze: bool = True):
    for p in model.t5.decoder.parameters():
        p.requires_grad = (not freeze)

def freeze_encoder(model: BottleneckT5AE, freeze: bool = False):
    for p in model.t5.encoder.parameters():
        p.requires_grad = (not freeze)


In [ ]:
freeze_decoder(model, freeze=True)

In [ ]:
val_ratio = 0.02
val_size = int(len(ds) * val_ratio)
train_size = len(ds) - val_size
train_ds, val_ds = random_split(ds, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

train_size, val_size


(263579, 5379)

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler('cuda')


In [ ]:
@torch.no_grad()
def reconstruct_samples(texts: List[str], max_new_tokens=128):
    model.eval()
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)

    mem, mem_mask, _ = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
    enc_out = BaseModelOutput(last_hidden_state=mem)

    #beam search
    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
        early_stopping=True,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)



# quick sanity preview before training
sample_texts = [ds[i] for i in random.sample(range(len(ds)), 3)]
print("ORIGINAL:\n", "\n---\n".join(sample_texts))
print("\nRECON (untrained bottleneck layers):")
print("\n---\n".join(reconstruct_samples(sample_texts)))


ORIGINAL:
 This one hardly compares to the space adventures of its time. Those being Star Wars and Star Trek. And while I am no fan of Star Trek, I recognize that this film pales in comparison to the series Trekkies ooze over.
---
It may be a remake of the 1937 film by Capra, but it is wrong to consider it only in that way! It was supposed to expose Hilton's novel in a completely different way. As a musical is excellent.
---
I may be taking it too seriously, and if that's the case I can at least say that it's superbly made, extremely entertaining (and pretty mature, too), and with an ambiance like no other.

RECON (untrained bottleneck layers):
a a a a a a a a the. Then there is a chance to find out more about it. a a., a a a a
---
a a a a a a a a the. Then there is a chance to find out more about it. a a., a a a a
---
a a a a a a a a the. Then there is a chance to find out more about it. a a., a a a a


In [ ]:
@torch.no_grad()
def encode_texts_to_zslots(texts: List[str]):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
    _, _, z_slots = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
    return z_slots  # [B,K,latent]

In [ ]:
def run_eval():
    model.eval()
    losses = []
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device == "cuda")):
            out = model(**batch)
            losses.append(out.loss.item())
    return sum(losses) / max(1, len(losses))



# ---- Cycle loss hyperparams ----
CYCLE_EVERY = 300       # compute cycle loss every N steps
CYCLE_BS = 2            # small batch for cycle
LAMBDA_CYCLE = 0.05     # cycle weight
CYCLE_MAX_NEW_TOKENS = 128

global_step = 0
best_val = float("inf")

DECODER_FREEZE_STEPS = 14000  # keep your choice


for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running = 0.0

    for batch_idx, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=(device == "cuda")):
            # ----- main AE loss -----
            out = model(**batch)
            loss = out.loss

            # ----- cycle consistency (occasionally) -----
            do_cycle = (global_step % CYCLE_EVERY == 0) and (batch["labels"].size(0) >= CYCLE_BS)

            if do_cycle:
                # Recover clean target text from labels (labels have -100 on pads)
                ids = batch["labels"][:CYCLE_BS].clone()
                ids[ids == -100] = tokenizer.pad_token_id
                texts_clean = tokenizer.batch_decode(ids, skip_special_tokens=True)

                # z1: encode original clean texts
                z1 = encode_texts_to_zslots(texts_clean)  # [B,K,latent]

                # Decode from bottleneck representation
                enc = tokenizer(
                    texts_clean,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_LEN
                ).to(device)

                # If your model has encode_to_mem (recommended)
                mem, mem_mask, _ = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
                enc_out = BaseModelOutput(last_hidden_state=mem)

                gen_ids = model.t5.generate(
                    encoder_outputs=enc_out,
                    attention_mask=mem_mask,
                    max_new_tokens=CYCLE_MAX_NEW_TOKENS,
                    do_sample=False
                )
                texts_hat = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

                # z2: encode reconstructed texts
                z2 = encode_texts_to_zslots(texts_hat)

                # cosine distance cycle loss
                z1n = F.normalize(z1, dim=-1)
                z2n = F.normalize(z2, dim=-1)
                cycle_loss = (1.0 - (z1n * z2n).sum(dim=-1)).mean()

                loss = loss + LAMBDA_CYCLE * cycle_loss

        # backward
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()

        # set noise for NEXT batch (collator runs before we get batch)
        next_step = global_step + 1
        if next_step < NOISE_WARMUP_STEPS:
            p = TARGET_NOISE_PROB * (next_step / float(NOISE_WARMUP_STEPS))
        else:
            p = TARGET_NOISE_PROB
        collate_fn.set_noise_prob(p)

        # ---- increment step counter ----
        global_step += 1

        # ---- decoder unfreeze point ----
        if global_step == DECODER_FREEZE_STEPS:
            print("Unfreezing T5 decoder...")
            freeze_decoder(model, freeze=False)

            optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LR,
                weight_decay=WEIGHT_DECAY
            )

            remaining_steps = total_steps - global_step
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(remaining_steps * WARMUP_RATIO),
                num_training_steps=remaining_steps
            )

        scheduler.step()

        running += loss.item()

        if global_step % 200 == 0:
            avg = running / 200
            running = 0.0
            print(f"epoch {epoch} | step {global_step} | train_loss {avg:.4f}")

        if global_step % 800 == 0:
            texts = [ds[i] for i in random.sample(range(len(ds)), 2)]
            outs = reconstruct_samples(texts)
            print("\n--- QUAL CHECK ---")
            for a, b in zip(texts, outs):
                print("IN :", a[:250])
                print("OUT:", b[:250])
                print("")

    val_loss = run_eval()
    elapsed = time.time() - t0
    print(f"\n[EPOCH {epoch}] val_loss={val_loss:.4f} | time={elapsed/60:.1f} min")

    if val_loss < best_val:
        best_val = val_loss
        ckpt_path = os.path.join(OUT_DIR, "best.pt")
        torch.save(model.state_dict(), ckpt_path)
        print(f"Saved best checkpoint to: {ckpt_path}")

print("Done. Best val loss:", best_val)


epoch 1 | step 200 | train_loss 4.4938
epoch 1 | step 400 | train_loss 4.2615
epoch 1 | step 600 | train_loss 4.1264
epoch 1 | step 800 | train_loss 4.0208

--- QUAL CHECK ---
IN : and sympathetic as Anna's father.
OUT: as a sympathetic character.

IN : characters that were purposefully badly done in CGI to make it look like they were from a game, and who were OBVIOUSLY stolen from Japanese horror movies. To be honest, it was hilariously bad, and something I'd expect from a midnight showing of a mad
OUT: I was able to see this movie as a horror horror movie, but it was a shame that it was not filmed in a movie with a sleeve-like screenplay. This is a horrible horror movie.

epoch 1 | step 1000 | train_loss 3.6546
epoch 1 | step 1200 | train_loss 3.5207
epoch 1 | step 1400 | train_loss 3.3955
epoch 1 | step 1600 | train_loss 3.3254

--- QUAL CHECK ---
IN : And in this instance the actors peg out in exactly the order that everyone expects them to - i quickly wrote a list after being intr

In [ ]:
from google.colab import files

files.download('/content/ae_story1_t5tok_bottleneck256/best.pt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================
# SINGLE-CELL AE EVAL SUITE
# Works with your current code objects:
# - model: BottleneckT5AE
# - tokenizer: T5TokenizerFast
# - ds: ChunkTextDataset (strings)
# - device
# Checkpoint: /content/ae_story1_t5tok_bottleneck256/best.pt
# ============================

import os, re, math, random
import numpy as np
import torch
import torch.nn.functional as F
from transformers.modeling_outputs import BaseModelOutput

# --------- config ----------
CKPT_PATH = "/content/ae_story1_t5tok_bottleneck256/best.pt"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

N_SAMPLES = 64         # eval set size (keep small for T4)
MAX_LEN = 96
MAX_NEW_TOKENS = 128

BEAMS = 4
NO_REPEAT_NGRAM = 3
REP_PENALTY = 1.15

# corruption sweep (applied at eval-time)
NOISE_PS = [0.0, 0.05, 0.10, 0.15, 0.25]

# Needle test
NEEDLE = "ZXQ-9281-HELLO-KITTEN-42"
NEEDLE_N = 12

# --------- load checkpoint ----------
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
_ = model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu"), strict=True)
model.to(device).eval()
print("Loaded checkpoint:", CKPT_PATH)

# --------- helpers ----------
def _truncate_to_len(x, m):
    return x[:m] if len(x) > m else x

def _tokenize_texts(texts):
    return tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to(device)

@torch.no_grad()
def reconstruct(texts, do_sample=False):
    enc = _tokenize_texts(texts)
    mem, mem_mask, _ = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
    enc_out = BaseModelOutput(last_hidden_state=mem)
    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=do_sample,
        num_beams=BEAMS if not do_sample else 1,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        repetition_penalty=REP_PENALTY,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

@torch.no_grad()
def reconstruct_with_input_corruption(texts, noise_p):
    """
    Corrupt the *input_ids* at eval time by setting random attended tokens to PAD.
    Labels stay implicit (we're just measuring recon).
    """
    enc = _tokenize_texts(texts)
    input_ids = enc["input_ids"].clone()
    attn = enc["attention_mask"]

    if noise_p > 0:
        corrupt = (torch.rand_like(input_ids.float()) < noise_p) & (attn == 1)
        corrupt &= (input_ids != tokenizer.pad_token_id)
        input_ids[corrupt] = tokenizer.pad_token_id

    mem, mem_mask, _ = model.encode_to_mem(input_ids, attn)
    enc_out = BaseModelOutput(last_hidden_state=mem)

    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=BEAMS,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        repetition_penalty=REP_PENALTY,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

def token_level_scores(inp_texts, out_texts):
    """
    Returns:
      - token_exact_match_ratio (aligned by position)
      - length_ratio (len_out/len_in)
      - mean_abs_len_diff
    """
    in_ids = tokenizer(inp_texts, add_special_tokens=False).input_ids
    out_ids = tokenizer(out_texts, add_special_tokens=False).input_ids

    matches = []
    len_ratios = []
    abs_diffs = []

    for a, b in zip(in_ids, out_ids):
        la, lb = len(a), len(b)
        m = min(la, lb)
        if m == 0:
            matches.append(0.0)
        else:
            eq = sum(1 for i in range(m) if a[i] == b[i])
            matches.append(eq / m)
        len_ratios.append((lb / la) if la > 0 else 0.0)
        abs_diffs.append(abs(lb - la))

    return float(np.mean(matches)), float(np.mean(len_ratios)), float(np.mean(abs_diffs))

def levenshtein_norm(a, b):
    """
    Normalized Levenshtein distance on characters (distance / max_len).
    Small is good.
    """
    a = a or ""
    b = b or ""
    n, m = len(a), len(b)
    if max(n, m) == 0:
        return 0.0
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = cur
    return dp[m] / max(n, m)

def avg_norm_levenshtein(inp_texts, out_texts):
    return float(np.mean([levenshtein_norm(a, b) for a, b in zip(inp_texts, out_texts)]))

@torch.no_grad()
def bottleneck_ablation(texts, mode="zero"):
    """
    mode:
      - "zero": mem becomes all zeros (content removed)
      - "emb_only": keep only mem_slot_emb broadcast (no content)
      - "shuffle": shuffle mem across batch
    """
    enc = _tokenize_texts(texts)
    mem, mem_mask, _ = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])

    if mode == "zero":
        mem2 = torch.zeros_like(mem)
    elif mode == "emb_only":
        mem2 = model.mem_slot_emb[None, :, :].expand(mem.size(0), -1, -1).to(mem.dtype).to(mem.device)
    elif mode == "shuffle":
        perm = torch.randperm(mem.size(0), device=mem.device)
        mem2 = mem[perm]
    else:
        raise ValueError("unknown mode")

    enc_out = BaseModelOutput(last_hidden_state=mem2)
    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=BEAMS,
        no_repeat_ngram_size=NO_REPEAT_NGRAM,
        repetition_penalty=REP_PENALTY,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

@torch.no_grad()
def slot_stats(texts):
    enc = _tokenize_texts(texts)
    _, _, z = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])  # [B,K,latent]
    # variance across batch per slot
    var_per_slot = z.var(dim=0).mean(dim=-1)  # [K]
    # within-sample slot similarity (collapse check)
    z_n = F.normalize(z, dim=-1)
    B, K, D = z_n.shape
    sims = []
    for i in range(B):
        # average off-diagonal cosine similarity
        mat = z_n[i] @ z_n[i].T  # [K,K]
        off = (mat.sum() - mat.diag().sum()) / (K*K - K)
        sims.append(off.item())
    return var_per_slot.detach().cpu().numpy(), float(np.mean(sims))

# --------- sample texts ----------
idxs = random.sample(range(len(ds)), k=min(N_SAMPLES, len(ds)))
base_texts = [ds[i] for i in idxs]

print(f"Eval samples: {len(base_texts)}")

# ============================
# TEST A: Identity mapping (p=0)
# ============================
outs_p0 = reconstruct_with_input_corruption(base_texts, noise_p=0.0)
temr0, lratio0, lendiff0 = token_level_scores(base_texts, outs_p0)
lev0 = avg_norm_levenshtein(base_texts, outs_p0)

print("\n[TEST A] Identity mapping (no corruption)")
print(f" token_exact_match_ratio: {temr0:.4f} (higher=better)")
print(f" norm_levenshtein:        {lev0:.4f} (lower=better)")
print(f" len_out/len_in:          {lratio0:.3f} (closer to 1=better)")
print(f" mean_abs_len_diff:       {lendiff0:.2f}")

# ============================
# TEST B: Corruption sweep
# ============================
print("\n[TEST B] Corruption sweep (PAD masking at eval)")
for p in NOISE_PS:
    outs = reconstruct_with_input_corruption(base_texts, noise_p=p)
    temr, lratio, lendiff = token_level_scores(base_texts, outs)
    lev = avg_norm_levenshtein(base_texts, outs)
    print(f" p={p:>4.2f} | token_match={temr:.4f} | lev={lev:.4f} | len_ratio={lratio:.3f} | abs_len_diff={lendiff:.2f}")

# ============================
# TEST C: Needle preservation
# ============================
needle_texts = []
for _ in range(NEEDLE_N):
    t = random.choice(base_texts)
    # insert needle near the middle
    mid = len(t)//2
    t2 = t[:mid] + f" {NEEDLE} " + t[mid:]
    needle_texts.append(t2)

outs_needle = reconstruct_with_input_corruption(needle_texts, noise_p=0.0)
needle_hit = sum(1 for o in outs_needle if NEEDLE in o)
print("\n[TEST C] Needle preservation (noise=0)")
print(f" needle_hit_rate: {needle_hit}/{len(needle_texts)} = {needle_hit/len(needle_texts):.2%} (want ~100% for true copy AE)")

# ============================
# TEST D: Bottleneck ablations
# ============================
outs_zero = bottleneck_ablation(base_texts[:8], mode="zero")
outs_emb  = bottleneck_ablation(base_texts[:8], mode="emb_only")
outs_shuf = bottleneck_ablation(base_texts[:8], mode="shuffle")

def quick_overlap(a, b):
    # rough token overlap (% of unique tokens in out that appear in in)
    a_t = set(re.findall(r"\w+", a.lower()))
    b_t = set(re.findall(r"\w+", b.lower()))
    if len(b_t) == 0: return 0.0
    return len(a_t & b_t) / len(b_t)

print("\n[TEST D] Bottleneck ablation quick read (8 examples)")
for i in range(2):  # print 2 examples to keep output short
    inp = base_texts[i]
    base = outs_p0[i]
    zro = outs_zero[i]
    emb = outs_emb[i]
    shf = outs_shuf[i]
    print(f"\nExample {i+1}")
    print("IN  :", inp[:220])
    print("OUT :", base[:220], f"| overlap={quick_overlap(inp, base):.2f}")
    print("ZERO:", zro[:220],  f"| overlap={quick_overlap(inp, zro):.2f}")
    print("EMB :", emb[:220],  f"| overlap={quick_overlap(inp, emb):.2f}")
    print("SHUF:", shf[:220],  f"| overlap={quick_overlap(inp, shf):.2f}")

# ============================
# TEST E: Slot collapse stats
# ============================
var_per_slot, mean_offdiag_sim = slot_stats(base_texts[:32])
print("\n[TEST E] Slot usage / collapse")
print(f" mean off-diag cosine(sim) between slots within a sample: {mean_offdiag_sim:.4f} (lower=better; high => collapse)")
print(f" per-slot variance (first 8 slots): {np.round(var_per_slot[:8], 4)}")

# ============================
# Pretty side-by-side (2 random)
# ============================
print("\n[QUAL] Side-by-side (2 random, p=0)")
for j in random.sample(range(len(base_texts)), k=2):
    print("\nIN :", base_texts[j][:260])
    print("OUT:", outs_p0[j][:260])


Loaded checkpoint: /content/ae_story1_t5tok_bottleneck256/best.pt
Eval samples: 64

[TEST A] Identity mapping (no corruption)
 token_exact_match_ratio: 0.4166 (higher=better)
 norm_levenshtein:        0.3116 (lower=better)
 len_out/len_in:          0.855 (closer to 1=better)
 mean_abs_len_diff:       12.84

[TEST B] Corruption sweep (PAD masking at eval)
 p=0.00 | token_match=0.4166 | lev=0.3116 | len_ratio=0.855 | abs_len_diff=12.84
 p=0.05 | token_match=0.3497 | lev=0.3283 | len_ratio=0.856 | abs_len_diff=13.05
 p=0.10 | token_match=0.3194 | lev=0.3623 | len_ratio=0.855 | abs_len_diff=13.20
 p=0.15 | token_match=0.2930 | lev=0.3836 | len_ratio=0.853 | abs_len_diff=12.95
 p=0.25 | token_match=0.2358 | lev=0.4363 | len_ratio=0.841 | abs_len_diff=13.53

[TEST C] Needle preservation (noise=0)
 needle_hit_rate: 0/12 = 0.00% (want ~100% for true copy AE)

[TEST D] Bottleneck ablation quick read (8 examples)

Example 1
IN  : This one hardly compares to the space adventures of its time. Thos

In [ ]:
# ============================
# SEMANTIC / LATENT-SPACE TESTS for your BottleneckT5AE
# Tests:
# 1) Sentiment separability in latent space (kNN + linear probe)
# 2) Local continuity (small edit => small latent change)
# 3) Interpolation path (z lerp) => smooth semantic changes? + monotonicity
# 4) Retrieval sanity: nearest neighbors in latent space are semantically similar
# ============================

import random, re, math
import numpy as np
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MAX_LEN = 192
N = 800          # how many examples to test (keep <= 2000 on T4)
KNN_K = 5

# ---- helper tokenize ----
def tok(texts):
    return tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)

@torch.no_grad()
def encode_z(texts):
    """
    returns z_slots: [B,K,latent] and pooled z: [B, K*latent] and mean z: [B, latent]
    """
    enc = tok(texts)
    _, _, z = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
    z = z.float()
    z_flat = z.reshape(z.size(0), -1)             # [B, K*latent]
    z_mean = z.mean(dim=1)                        # [B, latent]
    return z, z_flat, z_mean

def cosine_mat(A, B):
    A = F.normalize(A, dim=-1)
    B = F.normalize(B, dim=-1)
    return A @ B.T

# ---- pick data ----
# Your ds is chunks; sentiment labels are in original df but ds doesn't store them.
# We'll approximate sentiment by sampling directly from df if it exists.
# If df isn't in memory, we fallback to ds-only (then we skip sentiment tests).
has_df = "df" in globals() and ("sentiment" in df.columns) and ("review" in df.columns)

def sample_labeled(n):
    sub = df.sample(n=n, random_state=SEED).copy()
    texts = sub["review"].astype(str).tolist()
    # clean html similar to your pipeline, but light
    texts = [re.sub(r"<br\s*/?>", " ", t) for t in texts]
    texts = [re.sub(r"<[^>]+>", " ", t) for t in texts]
    texts = [re.sub(r"\s+", " ", t).strip() for t in texts]
    y = (sub["sentiment"].values == "positive").astype(np.int64)
    return texts, y

def sample_unlabeled(n):
    idxs = random.sample(range(len(ds)), k=min(n, len(ds)))
    return [ds[i] for i in idxs]

# ============================
# TEST 1: Sentiment separability (kNN + linear probe)
# ============================
print("\n[TEST 1] Sentiment separability in latent space")
if not has_df:
    print("df with sentiment not found in this runtime -> skipping Test 1.")
else:
    texts, y = sample_labeled(N)
    _, Zflat, Zmean = encode_z(texts)

    # Split train/test
    perm = np.random.permutation(len(texts))
    split = int(0.8 * len(texts))
    tr, te = perm[:split], perm[split:]

    Ztr = Zflat[tr].detach().cpu()
    ytr = torch.tensor(y[tr])
    Zte = Zflat[te].detach().cpu()
    yte = torch.tensor(y[te])

    # kNN classification in latent space
    # similarity test: for each test, pick topK train by cosine, majority vote
    sims = cosine_mat(Zte, Ztr)          # [Te, Tr]
    topk = sims.topk(KNN_K, dim=1).indices
    preds = []
    for i in range(topk.size(0)):
        votes = ytr[topk[i]].numpy()
        preds.append(int(votes.mean() >= 0.5))
    preds = np.array(preds)
    acc_knn = (preds == yte.numpy()).mean()

    # simple linear probe (logistic regression) via torch (fast enough)
    W = torch.zeros(Ztr.size(1), 1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], max_iter=50)

    ytr_f = ytr.float().unsqueeze(1)
    def closure():
        opt.zero_grad()
        logits = Ztr @ W + b
        loss = F.binary_cross_entropy_with_logits(logits, ytr_f)
        loss.backward()
        return loss

    opt.step(closure)
    with torch.no_grad():
        logits = Zte @ W + b
        preds_lp = (torch.sigmoid(logits).squeeze(1) >= 0.5).long()
        acc_lp = (preds_lp == yte).float().mean().item()

    print(f" kNN (k={KNN_K}) acc: {acc_knn:.3f}")
    print(f" Linear probe acc:   {acc_lp:.3f}")
    print(" Interpretation: >0.65 is usually decent signal; >0.75 is strong for this setup.")

# ============================
# TEST 2: Local continuity (small edit => small latent change)
# ============================
print("\n[TEST 2] Local continuity under small edits")
base = sample_unlabeled(64)
# create lightly edited versions: delete 10% words + swap 2 adjacent words
edited = []
for t in base:
    words = t.split()
    if len(words) < 8:
        edited.append(t)
        continue
    # delete ~10%
    keep = [w for i,w in enumerate(words) if random.random() > 0.10]
    if len(keep) < 5: keep = words
    # swap 2 adjacent
    j = random.randint(0, len(keep)-2)
    keep[j], keep[j+1] = keep[j+1], keep[j]
    edited.append(" ".join(keep))

_, Zb, Zbmean = encode_z(base)
_, Ze, Zemean = encode_z(edited)

# cosine similarity base vs edited (higher means continuity)
sim_flat = F.cosine_similarity(Zb, Ze, dim=-1).mean().item()
sim_mean = F.cosine_similarity(Zbmean, Zemean, dim=-1).mean().item()

print(f" cosine(z_flat(base), z_flat(edited)) mean: {sim_flat:.3f}")
print(f" cosine(z_mean(base), z_mean(edited)) mean: {sim_mean:.3f}")
print(" Interpretation: ~0.7+ suggests smoothness; ~0.4-0.6 is moderate; <0.3 is unstable.")

# ============================
# TEST 3: Interpolation (z lerp) + semantic monotonicity via sentiment probe (optional)
# ============================
print("\n[TEST 3] Latent interpolation sanity (decode from mixed memory)")
# We'll mix two examples in latent slot space and decode.
# NOTE: this bypasses re-encoding; it tests whether latent space is continuous.
@torch.no_grad()
def decode_from_zslots(z_slots):
    # z_slots: [B,K,latent]
    pooled_rec = model.up(z_slots)  # [B,K,d_model]
    flat = pooled_rec.reshape(pooled_rec.size(0), -1)
    mem_flat = model.to_mem(flat)
    mem = mem_flat.view(mem_flat.size(0), model.bottleneck_seq_len, model.d_model)
    mem = mem + model.mem_slot_emb[None, :, :]
    mem_mask = torch.ones(mem.size(0), mem.size(1), dtype=torch.long, device=mem.device)
    enc_out = BaseModelOutput(last_hidden_state=mem)
    gen = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=96,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
        early_stopping=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

# choose two random texts
a, b = random.sample(sample_unlabeled(200), 2)
za, _, _ = encode_z([a])
zb, _, _ = encode_z([b])
za = za.to(device); zb = zb.to(device)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
mix = torch.cat([(1-t)*za + t*zb for t in alphas], dim=0)
outs_mix = decode_from_zslots(mix)

print("\nA:", a[:220])
print("B:", b[:220])
for t, o in zip(alphas, outs_mix):
    print(f"\nalpha={t:.2f} OUT:", o[:220])

print("\nInterpretation: if alpha=0 looks like A and alpha=1 looks like B, and middle outputs blend topics smoothly, your latent is continuous.")

# ============================
# TEST 4: Nearest-neighbor retrieval (semantic neighbors)
# ============================
print("\n[TEST 4] Nearest neighbors in latent space")
pool = sample_unlabeled(300)
_, Zp, _ = encode_z(pool)
Zp_cpu = Zp.detach().cpu()

q_idx = random.randint(0, len(pool)-1)
q_text = pool[q_idx]
q_vec = Zp_cpu[q_idx:q_idx+1]

sims = cosine_mat(q_vec, Zp_cpu).squeeze(0)
top = torch.topk(sims, k=6).indices.tolist()  # includes itself
top = [i for i in top if i != q_idx][:5]

print("\nQUERY:", q_text[:260])
for rank, i in enumerate(top, 1):
    print(f"\nNN{rank} sim={sims[i].item():.3f}:", pool[i][:260])



[TEST 1] Sentiment separability in latent space
 kNN (k=5) acc: 0.637
 Linear probe acc:   0.781
 Interpretation: >0.65 is usually decent signal; >0.75 is strong for this setup.

[TEST 2] Local continuity under small edits
 cosine(z_flat(base), z_flat(edited)) mean: 0.908
 cosine(z_mean(base), z_mean(edited)) mean: 0.952
 Interpretation: ~0.7+ suggests smoothness; ~0.4-0.6 is moderate; <0.3 is unstable.

[TEST 3] Latent interpolation sanity (decode from mixed memory)

A: Do we see her? No. Does Ajax go off to find her as soon as he hears this? No. Now THAT's a marriage! Amidst all this irritatingly puerile crap, some website described this film as " a cross between Jackie Chan & Guy Ritc
B: It's rather shocking stuff, but if you had the opportunity for the Q&A sessions after the screenings, it clearly opens up a bag of worms that leaves you wondering whether this is art or just the lowest common denominator

alpha=0.00 OUT: Do we see this as a marriage? No, no. A website goes on to se

In [ ]:
# ============================
# TEST 7: Sentiment preservation through decode (Input vs Output)
# Requirements:
# - model: BottleneckT5AE
# - tokenizer: T5TokenizerFast
# - ds: list-like dataset of strings (your chunks)
# - device
# - checkpoint path exists
# ============================

import random, re, math, numpy as np, torch
from transformers import pipeline

CKPT = "/content/ae_story1_t5tok_bottleneck256/best.pt"
N_SAMPLES_TOTAL = 200          # increase to 500 if you want stronger stats (slower)
MAX_NEW_TOKENS = 64
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- load checkpoint
sd = torch.load(CKPT, map_location="cpu")
model.load_state_dict(sd, strict=True)
model.to(device)
model.eval()
print(f"Loaded checkpoint: {CKPT}")

# ---- deterministic AE decode (beam)
from transformers.modeling_outputs import BaseModelOutput

@torch.no_grad()
def ae_decode(texts, max_new_tokens=MAX_NEW_TOKENS):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=192).to(device)
    mem, mem_mask, _ = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])
    enc_out = BaseModelOutput(last_hidden_state=mem)

    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
        early_stopping=True,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

# ---- sentiment classifier (fast, works well; not IMDB-specific but good enough for this test)
clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1,
)

def normalize_label(x):
    x = x.upper()
    if "POS" in x: return "pos"
    if "NEG" in x: return "neg"
    return x.lower()

# ---- build eval set from your original df (IMDB has labels)
# We assume you still have df with columns: review, sentiment
# But your ds contains chunks (no labels). We'll label by mapping chunk -> original review sentiment
# Simple approach: sample from df directly and feed through your chunker? Too heavy.
# Better: use df directly for semantics test (works even if AE trained on chunks).
# We'll just sample from df[review], run AE on first 192 tokens worth.

assert "df" in globals(), "df not found. Make sure your IMDB dataframe is loaded as df."
assert "review" in df.columns and "sentiment" in df.columns, "df must have columns: review, sentiment"

# balanced sampling
pos_df = df[df["sentiment"].str.lower().str.contains("pos")]
neg_df = df[df["sentiment"].str.lower().str.contains("neg")]

n_half = N_SAMPLES_TOTAL // 2
pos_texts = pos_df["review"].astype(str).sample(n_half, random_state=SEED).tolist()
neg_texts = neg_df["review"].astype(str).sample(n_half, random_state=SEED).tolist()

# clean HTML like your pipeline
def clean_html(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

pos_texts = [clean_html(t) for t in pos_texts]
neg_texts = [clean_html(t) for t in neg_texts]

texts_in = pos_texts + neg_texts
y_true = (["pos"] * n_half) + (["neg"] * n_half)

# ---- AE decode in batches
B = 16
texts_out = []
for i in range(0, len(texts_in), B):
    batch = texts_in[i:i+B]
    texts_out.extend(ae_decode(batch))

# ---- classify input + output (batch for speed)
def run_clf(texts, batch_size=32):
    preds = []
    scores = []
    for i in range(0, len(texts), batch_size):
        out = clf(texts[i:i+batch_size], truncation=True)
        preds.extend([normalize_label(o["label"]) for o in out])
        scores.extend([float(o["score"]) for o in out])
    return preds, np.array(scores)

pred_in,  score_in  = run_clf(texts_in)
pred_out, score_out = run_clf(texts_out)

# ---- metrics
y_true_arr = np.array(y_true)
pred_in_arr = np.array(pred_in)
pred_out_arr = np.array(pred_out)

acc_in  = (pred_in_arr == y_true_arr).mean()
acc_out = (pred_out_arr == y_true_arr).mean()
agree_in_out = (pred_in_arr == pred_out_arr).mean()

conf_drop = np.mean(score_in - score_out)

print("\n[TEST 7] Sentiment preservation through decode")
print(f"Classifier accuracy on INPUT  vs true label: {acc_in:.3f}")
print(f"Classifier accuracy on OUTPUT vs true label: {acc_out:.3f}")
print(f"Agreement: classifier(INPUT) == classifier(OUTPUT): {agree_in_out:.3f}")
print(f"Mean confidence drop (input_score - output_score): {conf_drop:.3f}")

# ---- show flip cases (where classifier changes label after AE)
flip_idxs = np.where(pred_in_arr != pred_out_arr)[0]
print(f"\nFlips: {len(flip_idxs)}/{len(texts_in)} = {len(flip_idxs)/len(texts_in):.2%}")

# show a few examples
show_k = 6
if len(flip_idxs) > 0:
    print("\nExamples of flips:")
    for j in flip_idxs[:show_k]:
        print("="*80)
        print(f"TRUE: {y_true_arr[j]} | IN_PRED: {pred_in_arr[j]} ({score_in[j]:.2f}) | OUT_PRED: {pred_out_arr[j]} ({score_out[j]:.2f})")
        print("IN :", texts_in[j][:300])
        print("OUT:", texts_out[j][:300])
else:
    print("\nNo flips in this sample (good sign).")


Loaded checkpoint: /content/ae_story1_t5tok_bottleneck256/best.pt


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



[TEST 7] Sentiment preservation through decode
Classifier accuracy on INPUT  vs true label: 0.910
Classifier accuracy on OUTPUT vs true label: 0.700
Agreement: classifier(INPUT) == classifier(OUTPUT): 0.710
Mean confidence drop (input_score - output_score): 0.013

Flips: 58/200 = 29.00%

Examples of flips:
TRUE: pos | IN_PRED: pos (0.99) | OUT_PRED: neg (0.99)
IN : What can you say about a short little film filled with secondary actors and no "stars", with absolutely no bad performances, not one poorly delivered line of dialog, and with the cold violence on one hand being balanced by the warm "heart" of the protagonists in the story? It's a jewel, a diamond, a
OUT: What can you say about this film without a full cast of poor actors, no short shorts, no direct performances of "hot," poorly balanced characters, and a jewel in the heart of the Western dialect, not one that evokes the spirit of a diamond? It's a story of
TRUE: pos | IN_PRED: pos (1.00) | OUT_PRED: neg (0.94)
IN : "The Cou

In [ ]:
# ============================
# TEST 8: Latent sentiment direction (arithmetic) - CODE CELL
# Requires: model, tokenizer, device, MAX_LEN, BOTTLENECK_SEQ_LEN
# Uses: IMDB Dataset.csv or df (review, sentiment)
# ============================

import os, random, re
import numpy as np
import torch
import torch.nn.functional as F
from transformers import pipeline
from transformers.modeling_outputs import BaseModelOutput

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# 0) Load labeled data (df)
# ----------------------------
try:
    df  # noqa
    print("[OK] df already exists.")
except NameError:
    import pandas as pd
    if os.path.exists("IMDB Dataset.csv"):
        df = pd.read_csv("IMDB Dataset.csv")
        print("[OK] Loaded df from IMDB Dataset.csv")
    elif os.path.exists("/content/IMDB Dataset.csv"):
        df = pd.read_csv("/content/IMDB Dataset.csv")
        print("[OK] Loaded df from /content/IMDB Dataset.csv")
    else:
        raise FileNotFoundError("Could not find IMDB Dataset.csv and df is not defined.")

# Normalize columns
assert "review" in df.columns and "sentiment" in df.columns, "df must have columns: review, sentiment"
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df["review"] = df["review"].astype(str)

# quick clean html (optional)
def clean_html(t: str) -> str:
    t = re.sub(r"<br\s*/?>", " ", t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["review_clean"] = df["review"].map(clean_html)

# ----------------------------
# 1) Sentiment classifier (fast, on GPU if available)
# ----------------------------
clf_device = 0 if torch.cuda.is_available() else -1
sent_clf = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=clf_device
)

def clf_score(texts, batch_size=16):
    """
    Returns: list of (label, prob_pos) where prob_pos is P(POSITIVE)
    """
    outs = sent_clf(texts, truncation=True, batch_size=batch_size)
    res = []
    for o in outs:
        label = o["label"]
        score = float(o["score"])
        prob_pos = score if label == "POSITIVE" else 1.0 - score
        res.append((label, prob_pos))
    return res

# ----------------------------
# 2) Encode text -> z_slots, then z_mean (stable)
# ----------------------------
@torch.no_grad()
def encode_to_zslots(texts, batch_size=16):
    model.eval()
    Z = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        ).to(device)
        _, _, z_slots = model.encode_to_mem(enc["input_ids"], enc["attention_mask"])  # [B,K,latent]
        Z.append(z_slots.detach().float().cpu())
    return torch.cat(Z, dim=0)  # [N,K,latent]

def zslots_to_zmean(z_slots):
    # z_slots: [N,K,latent] -> [N,latent]
    return z_slots.mean(dim=1)

# ----------------------------
# 3) Decode from z_slots directly (bypass encoder)
#    We replicate the part of encode_to_mem after z_slots:
#    pooled_rec = up(z_slots) -> to_mem -> add mem_slot_emb -> generate
# ----------------------------
@torch.no_grad()
def decode_from_zslots(z_slots, max_new_tokens=128, num_beams=4):
    """
    z_slots: [B,K,latent_dim] on device
    returns list[str]
    """
    model.eval()
    B, K, latent = z_slots.shape

    pooled_rec = model.up(z_slots)  # [B,K,d_model]
    flat = pooled_rec.reshape(B, -1)  # [B, K*d_model]
    mem_flat = model.to_mem(flat)     # [B, M*d_model]
    mem = mem_flat.view(B, model.bottleneck_seq_len, model.d_model)  # [B,M,d_model]
    mem = mem + model.mem_slot_emb[None, :, :]  # slot identity
    mem_mask = torch.ones(B, model.bottleneck_seq_len, dtype=torch.long, device=z_slots.device)

    enc_out = BaseModelOutput(last_hidden_state=mem)

    gen_ids = model.t5.generate(
        encoder_outputs=enc_out,
        attention_mask=mem_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=num_beams,
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
        early_stopping=True,
    )
    return tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

# ----------------------------
# 4) Compute sentiment direction v in z_mean space
# ----------------------------
N_PER_CLASS = 400   # increase if you want (e.g., 800) but keep GPU time in mind
pos_texts = df[df["sentiment"] == "positive"]["review_clean"].tolist()
neg_texts = df[df["sentiment"] == "negative"]["review_clean"].tolist()
random.shuffle(pos_texts)
random.shuffle(neg_texts)
pos_texts = pos_texts[:N_PER_CLASS]
neg_texts = neg_texts[:N_PER_CLASS]

print(f"[INFO] Using {len(pos_texts)} positive + {len(neg_texts)} negative to build direction.")

z_pos_slots = encode_to_zslots(pos_texts, batch_size=16)  # cpu float
z_neg_slots = encode_to_zslots(neg_texts, batch_size=16)

z_pos_mean = zslots_to_zmean(z_pos_slots)  # [N,latent]
z_neg_mean = zslots_to_zmean(z_neg_slots)

v = (z_pos_mean.mean(dim=0) - z_neg_mean.mean(dim=0))  # [latent]
v = v / (v.norm(p=2) + 1e-9)  # normalize
print("[OK] Computed normalized sentiment direction v in z_mean space.")

# ----------------------------
# 5) Pick anchors and apply shifts
# ----------------------------
ANCHORS_PER_SIDE = 3
alpha_list = [0.0, 0.5, 1.0, 1.5, 2.0]

# pick anchors (texts)
neg_anchors = df[df["sentiment"] == "negative"]["review_clean"].sample(ANCHORS_PER_SIDE, random_state=SEED).tolist()
pos_anchors = df[df["sentiment"] == "positive"]["review_clean"].sample(ANCHORS_PER_SIDE, random_state=SEED+1).tolist()

def run_anchor_set(anchors, direction_sign=+1, title="NEG->POS"):
    """
    direction_sign=+1 means add +alpha*v (more positive)
    direction_sign=-1 means subtract alpha*v (more negative)
    """
    print("\n" + "="*90)
    print(f"[{title}] direction_sign={direction_sign} | alphas={alpha_list}")
    print("="*90)

    # encode anchors to z_slots
    z0_slots = encode_to_zslots(anchors, batch_size=ANCHORS_PER_SIDE)  # cpu
    z0_slots = z0_slots.to(device)  # [B,K,latent]

    # We'll shift each slot by the same v (broadcast) derived from z_mean direction
    v_dev = v.to(device)[None, None, :]  # [1,1,latent]

    for i, text in enumerate(anchors):
        print("\n" + "-"*90)
        print(f"Anchor {i+1} (snippet): {text[:220]}")

        # baseline decode at alpha=0
        z_slots_i = z0_slots[i:i+1]  # [1,K,latent]

        outs = []
        for a in alpha_list:
            z_shift = z_slots_i + direction_sign * float(a) * v_dev  # [1,K,latent]
            dec = decode_from_zslots(z_shift, max_new_tokens=128, num_beams=4)[0]
            outs.append((a, dec))

        # classifier on outputs
        clf_out = clf_score([o[1] for o in outs], batch_size=8)

        for (a, dec), (lab, ppos) in zip(outs, clf_out):
            print(f"\nalpha={a:>4} | clf={lab:>8} | P(pos)={ppos:.3f}")
            print(dec[:380])

# NEG -> POS (add +alpha*v)
run_anchor_set(neg_anchors, direction_sign=+1, title="NEG -> POS (add +alpha*v)")

# POS -> NEG (subtract alpha*v)
run_anchor_set(pos_anchors, direction_sign=-1, title="POS -> NEG (add -alpha*v)")

print("\n[Done] If P(pos) increases monotonically for NEG anchors as alpha rises (and decreases for POS anchors), Test 8 succeeded.")


[OK] df already exists.


Device set to use cuda:0


[INFO] Using 400 positive + 400 negative to build direction.
[OK] Computed normalized sentiment direction v in z_mean space.

[NEG -> POS (add +alpha*v)] direction_sign=1 | alphas=[0.0, 0.5, 1.0, 1.5, 2.0]

------------------------------------------------------------------------------------------
Anchor 1 (snippet): I was looking forward to seeing Bruce Willis in this, especially since I remember being mesmerised by the original when I was young. This movie is a perfect example of how movie companies can take a very good story and d

alpha= 0.0 | clf=POSITIVE | P(pos)=0.680
I was particularly looking forward to seeing Bruce Willis in particular, since I remember being meager when I was young. This movie is a perfect example of how a movie companies can take a good story and dumb it down until it's just another hype hype hyped it up against the fabled ridden American law/army system VS/Army, let's face it, the Russians make no Russian war over the

alpha= 0.5 | clf=POSITIVE | P(pos)=1.0

In [ ]:
# ===========================
# TEST 4c: Nearest-neighbors sanity check (z_mean)
# - 20 random queries
# - top-5 neighbors from a random pool (faster on T4)
# Requires: model, tokenizer, ds (ChunkTextDataset), device
# ===========================

import random, torch
import torch.nn.functional as F

model.eval()

# ---- knobs ----
N_QUERIES = 20
POOL_SIZE = 1500     # candidate pool size (increase if you want stronger NNs)
TOPK = 5
MAX_PRINT_CHARS = 260
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

def _short(s, n=MAX_PRINT_CHARS):
    s = " ".join(str(s).split())
    return s[:n] + ("..." if len(s) > n else "")

@torch.no_grad()
def encode_z_mean(texts, max_len=None):
    toks = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len if max_len is not None else getattr(tokenizer, "model_max_length", 512),
    ).to(device)

    mem, mem_mask, z_slots = model.encode_to_mem(
        input_ids=toks["input_ids"],
        attention_mask=toks.get("attention_mask", None),
    )

    z_mean = z_slots.mean(dim=1)  # [B, latent_dim]
    return F.normalize(z_mean.float(), dim=-1).detach().cpu()

# ---- build pool ----
all_texts = list(getattr(ds, "texts", ds))  # try ds.texts else assume ds is iterable of strings
N = len(all_texts)

pool_idx = random.sample(range(N), min(POOL_SIZE, N))
pool_texts = [all_texts[i] for i in pool_idx]

query_idx = random.sample(range(N), min(N_QUERIES, N))
query_texts = [all_texts[i] for i in query_idx]

print(f"Pool size: {len(pool_texts)} | Queries: {len(query_texts)} | TopK: {TOPK}")

# ---- encode ----
Z_pool = encode_z_mean(pool_texts)
Z_q = encode_z_mean(query_texts)

# ---- cosine sims + topk ----
S = (Z_q @ Z_pool.T)  # [Q, P]
topv, topi = torch.topk(S, k=TOPK, dim=1)

# ---- print ----
for qi in range(len(query_texts)):
    print("\n" + "="*92)
    print(f"QUERY #{qi+1}: {_short(query_texts[qi])}")
    for r in range(TOPK):
        pj = int(topi[qi, r].item())
        sim = float(topv[qi, r].item())
        print(f"  NN{r+1} sim={sim:.3f}: {_short(pool_texts[pj])}")

Pool size: 1500 | Queries: 20 | TopK: 5

QUERY #1: If you like porno-horror, this is your movie, otherwise stay away . (Adrienne fans will get to see her sagging breasts for a second or two)
  NN1 sim=0.511: But if you rather keep your sanity, stay AWAY.
  NN2 sim=0.429: He deserves better. If you don't know Abbie or his times, this movie won't help. This film lies.
  NN3 sim=0.416: You know where this is going. Overall the movie deserves four stars out of ten, and that's being generous. For all its misgivings, the musical score is well done.
  NN4 sim=0.414: If you like classical music or modern dance this could be your date movie. But otherwise one and half hour is just too long time. If you like to see skillful dancing in silver screen it's better to see Bollywood movie.
  NN5 sim=0.414: You should never see this show If you see it on. TURN IT OFF. Or you be cringing for the next 30 minutes.

QUERY #2: Her performance consists mostly of looking sad and morose while mourning the loss